12/09/2026
First version

In [2]:
from io import TextIOWrapper
from pathlib import Path

import chess
import chess.engine
import chess.pgn
import numpy as np
import pandas as pd
import zstandard as zstd
from sklearn.tree import DecisionTreeClassifier


In [3]:
PGN_ZST_PATH = Path("DB/lichess_db_standard_rated_2013-01.pgn.zst")


def partidas_en_stream(ruta: Path):
    """Genera partidas PGN una a una desde un archivo .pgn.zst."""
    with ruta.open("rb") as archivo_comprimido:
        descompresor = zstd.ZstdDecompressor()
        with descompresor.stream_reader(archivo_comprimido) as flujo_binario:
            flujo_texto = TextIOWrapper(flujo_binario, encoding="utf-8")
            while partida := chess.pgn.read_game(flujo_texto):
                yield partida


def partida_en_indice(ruta: Path, indice: int):
    """Devuelve la partida con índice cero-based sin cargar todas las partidas."""
    if indice < 0:
        raise ValueError("El índice debe ser mayor o igual que cero")

    for indice_actual, partida in enumerate(partidas_en_stream(ruta)):
        if indice_actual == indice:
            return partida

    raise IndexError(f"No existe una partida con índice {indice}")


# INDICE_PARTIDA = 10
# partida_seleccionada = partida_en_indice(PGN_ZST_PATH, INDICE_PARTIDA)
# resultado_final = partida_seleccionada.headers.get("Result", "*")
# tablero = partida_seleccionada.board()
# movimientos_por_numero = {}

# for movimiento in partida_seleccionada.mainline_moves():
#     numero_movimiento = tablero.fullmove_number
#     notacion = tablero.san(movimiento)
#     movimientos_por_numero.setdefault(numero_movimiento, []).append(notacion)
#     tablero.push(movimiento)

# ultimos_10_movimientos = [
#     f"{numero}. {' '.join(movimientos)}"
#     for numero, movimientos in list(movimientos_por_numero.items())[-10:]
# ]

# print("Índice:", INDICE_PARTIDA)
# print("Resultado final:", resultado_final)
# print("Últimos 10 movimientos:")
# print(" ".join(ultimos_10_movimientos))

In [10]:
RESULTADO_A_CLASE = {
    "1-0": 0,
    "0-1": 1,
    "1/2-1/2": 2,
}

CARACTERISTICAS = [
    "reina_blancas",
    "reina_negras",
    "blancas_enrocadas",
    "negras_enrocadas",
    "dos_alfiles_blancas",
    "dos_alfiles_negras",
    "peon_pasado_blancas",
    "peon_pasado_negras",
]


def tiene_peon_pasado(tablero: chess.Board, color: chess.Color) -> bool:
    """Indica si el color tiene al menos un peón pasado."""
    peones = tablero.pieces(chess.PAWN, color)
    peones_rivales = tablero.pieces(chess.PAWN, not color)

    for casilla in peones:
        archivo = chess.square_file(casilla)
        rango = chess.square_rank(casilla)
        tiene_peon_rival_delante = any(
            abs(chess.square_file(casilla_rival) - archivo) <= 1
            and (
                chess.square_rank(casilla_rival) > rango
                if color == chess.WHITE
                else chess.square_rank(casilla_rival) < rango
            )
            for casilla_rival in peones_rivales
        )
        if not tiene_peon_rival_delante:
            return True

    return False


def estado_tablero(tablero: chess.Board, blancas_enrocadas: bool, negras_enrocadas: bool):
    """Extrae las características relevantes de una posición."""
    return {
        "reina_blancas": int(bool(tablero.pieces(chess.QUEEN, chess.WHITE))),
        "reina_negras": int(bool(tablero.pieces(chess.QUEEN, chess.BLACK))),
        "blancas_enrocadas": int(blancas_enrocadas),
        "negras_enrocadas": int(negras_enrocadas),
        "dos_alfiles_blancas": int(len(tablero.pieces(chess.BISHOP, chess.WHITE)) >= 2),
        "dos_alfiles_negras": int(len(tablero.pieces(chess.BISHOP, chess.BLACK)) >= 2),
        "peon_pasado_blancas": int(tiene_peon_pasado(tablero, chess.WHITE)),
        "peon_pasado_negras": int(tiene_peon_pasado(tablero, chess.BLACK)),
    }


def partida_a_dataframe(partida: chess.pgn.Game, movimientos_desde_final: int = 10):
    """Devuelve una fila con el estado 10 jugadas completas antes del final."""
    if movimientos_desde_final < 0:
        raise ValueError("movimientos_desde_final debe ser mayor o igual que cero")

    resultado = partida.headers.get("Result", "*")
    if resultado not in RESULTADO_A_CLASE:
        raise ValueError(f"Resultado PGN no válido para clasificación: {resultado}")

    movimientos = list(partida.mainline_moves())
    tablero = partida.board()
    estados = []
    blancas_enrocadas = False
    negras_enrocadas = False

    estados.append(estado_tablero(tablero, blancas_enrocadas, negras_enrocadas))

    for movimiento in movimientos:
        if tablero.is_castling(movimiento):
            if tablero.turn == chess.WHITE:
                blancas_enrocadas = True
            else:
                negras_enrocadas = True

        tablero.push(movimiento)
        estados.append(estado_tablero(tablero, blancas_enrocadas, negras_enrocadas))

    plies_desde_final = movimientos_desde_final * 2
    indice_estado = max(0, len(estados) - 1 - plies_desde_final)
    caracteristicas = estados[indice_estado]
    caracteristicas["y"] = RESULTADO_A_CLASE[resultado]

    return pd.DataFrame([caracteristicas], columns=CARACTERISTICAS + ["y"])


# INDICE_PARTIDA = 20
# partida_seleccionada = partida_en_indice(PGN_ZST_PATH, INDICE_PARTIDA)
# datos_partida = partida_a_dataframe(partida_seleccionada)
# X = datos_partida[CARACTERISTICAS]
# y = datos_partida["y"]

# print("Índice:", INDICE_PARTIDA)
# print("Resultado:", partida_seleccionada.headers["Result"])
# print("Forma de X:", X.shape)
# print("Forma de y:", y.shape)
# datos_partida

In [ ]:
def extrae_caracteristicas(ruta: Path, numero_partidas: int = 30):
    """Crea un DataFrame con las primeras partidas sin estados duplicados."""
    if numero_partidas < 0:
        raise ValueError("numero_partidas debe ser mayor o igual que cero")

    datos_partidas = []
    for indice, partida in enumerate(partidas_en_stream(ruta)):
        if indice >= numero_partidas:
            break
        datos_partidas.append(partida_a_dataframe(partida))

    if not datos_partidas:
        return pd.DataFrame(columns=CARACTERISTICAS + ["y"])

    return pd.concat(datos_partidas, ignore_index=True).drop_duplicates(
        subset=CARACTERISTICAS,
        ignore_index=True,
    )


datos_500_partidas = extrae_caracteristicas(PGN_ZST_PATH,1000)
datos_500_partidas

,reina_blancas,reina_negras,blancas_enrocadas,negras_enrocadas,dos_alfiles_blancas,dos_alfiles_negras,peon_pasado_blancas,peon_pasado_negras,y
0,1,1,0,0,1,1,0,0,0
1,1,1,0,0,1,0,0,0,0
2,0,0,0,0,0,0,0,1,1
3,1,1,1,0,1,1,0,0,1
4,1,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
120,1,1,0,1,1,0,1,1,1
121,0,0,1,0,0,1,1,0,1
122,0,0,1,1,1,0,0,1,1
123,1,1,0,1,1,0,0,1,0


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_500 = datos_500_partidas[CARACTERISTICAS]
y_500 = datos_500_partidas["y"]

X_train, X_validation, y_train, y_validation = train_test_split(
    X_500,
    y_500,
    test_size=0.2,
    random_state=42,
    stratify=y_500,
)

arbol_decision = DecisionTreeClassifier(random_state=42)
arbol_decision.fit(X_train, y_train)

predicciones_validation = arbol_decision.predict(X_validation)
precision_validation = accuracy_score(y_validation, predicciones_validation)

print(f"Precisión en validación: {precision_validation:.2%}")
precision_validation

Precisión en validación: 52.00%


0.52